In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "aGoodPassword!"
host = 'localhost' 
port = 27017 
database = 'aac' 
collection = 'animals'

# Connect to database via CRUD Module
db = AnimalShelter(username, password, host, port, database, collection)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Center(
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={
                    'width' : '300px',
                    'height' : 'auto'
                }
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),
    ),
    html.Center(html.P('Created by Peter Taylor')),
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
    
    html.Div(
        dcc.RadioItems(
            id='filter-type',
            options=[
                { 'label' : 'Water Rescue', 'value' : 'water' },
                { 'label' : 'Mountain or Wilderness Rescue', 'value' : 'mountain' },
                { 'label' : 'Disaster or Individual Tracking', 'value' : 'tracking' },
                { 'label' : 'Reset', 'value' : 'reset' }
            ],
            value='reset',
            labelStyle={
                'display' : 'flex',
                'width' : '100%'
            }
        ),
        style={'display' : 'flex'}
    ),
    
    html.Hr(),
    
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        filter_action = "native",
        sort_action = "native",
        sort_mode = "multi",
        column_selectable = False,
        row_selectable = "single",
        selected_rows = [0],
        page_action = "native",
        page_current = 0,
        page_size = 10
    ),
    
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################


@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    # setup the db query based on filter_type
    match filter_type:
        case 'water':
            query = {
                'animal_type' : 'Dog',
                'breed' : {'$in' : [
                    'Labrador Retriever',
                    'Chesapeake Bay Retriever',
                    'Newfoundland'
                ]},
                'sex_upon_outcome' : 'Intact Female',
                'age_upon_outcome_in_weeks' : {'$gte' : 26},
                'age_upon_outcome_in_weeks' : {'$lte' : 156}
            }
        case 'mountain':
            query = {
                'animal_type' : 'Dog',
                'breed' : {'$in' : [
                    'German Shepherd',
                    'Alaskan Malamute',
                    'Old English Sheepdog',
                    'Siberian Husky',
                    'Rottweiler'
                ]},
                'sex_upon_outcome' : 'Intact Male',
                'age_upon_outcome_in_weeks' : {'$gte' : 26},
                'age_upon_outcome_in_weeks' : {'$lte' : 156}
            }
        case 'tracking':
            query = {
                'animal_type' : 'Dog',
                'breed' : {'$in' : [
                    'Doberman Pinsch',
                    'German Shepherd',
                    'Golden Retriever',
                    'Bloodhound',
                    'Rottweiler'
                ]},
                'sex_upon_outcome' : 'Intact Male',
                'age_upon_outcome_in_weeks' : {'$gte' : 20},
                'age_upon_outcome_in_weeks' : {'$lte' : 300}
            }
        case 'reset':
            query = {}
        # default query if filter_type does not match any options
        case _:
            query = {}
            
    # execute query and create dataframe
    dff = pd.DataFrame.from_records(db.read(query))
    
    #Debug
    #print(len(df.to_dict(orient='records')))
    dff.drop(columns=['_id'],inplace=True)
    
    return dff.to_dict('records')


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # Input validation
    if viewData is None or len(viewData) == 0:
        return []
    
    # Create dataframe from input data
    dff = pd.DataFrame.from_dict(viewData)
    breed_counts = dff['breed'].value_counts().reset_index().head(25)
    breed_counts.columns = ['breed', 'count']
    
    # Defines bar chart
    fig = px.bar(
        breed_counts, x='breed', y='count', title='Animals by Breed (Top 25)'
    )
    
    # returns bar chart
    return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    selected_columns = selected_columns or [] # fixes callback iteration error when None
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None or len(viewData) == 0:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Because we only allow single row selection, the list can be converted to a row index here
    row = 0 if not index else index[0]
        
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
           # Austin TX is at [30.75,-97.48]
            center=[30.75,-97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
               
                # Marker with tool tip and popup
                # Column 13 and 14 define the grid-coordinates for 
                # the map
                dl.Marker(
                    position=[dff.iloc[row,13],dff.iloc[row,14]],
                    children=[
                        dl.Tooltip(dff.iloc[row,4]), # Column 4 defines the breed for the animal
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row,9]) # Column 9 defines the name of the animal
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://edgarlexicon-stellalithium-3000.codio.io/proxy/8050/
